In [10]:
import sys
from pathlib import Path

root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from langchain_core.messages import AIMessage, ToolMessage
from langchain_openai import ChatOpenAI

# Use the same OpenAI model for generation and evaluation during testing.
generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

from agent.agent import create_refund_agent

agent = create_refund_agent()

In [11]:
# Make one request to the agent.
user_input = "What is today's date? and get purchase history of user_10000000"

result = agent.invoke(
    {"messages": [{"role": "user", "content": user_input}]},
    config={"configurable": {"thread_id": "manual-tool-demo"}},
)

messages = result["messages"]

/Users/rahuldevbanjara/Documents/Personal Projects/AI-Agents-Eval/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
[INFO] tool=get_today_date
[INFO] tool=get_user_purchase_history user=user_10000000
/Users/rahuldevbanjara/Documents/Personal Projects/AI-Agents-Eval/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [12]:
messages

[HumanMessage(content="What is today's date?", additional_kwargs={}, response_metadata={}, id='4fb20bff-d000-4559-81bc-7253942abc40'),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_today_date', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_2330809596147589008': 'EmAKXgFpFH0Tittru2lxxCdlPpao07xKKdjmfHTjFe4cxKh4kY6tF+s2shjdNK+LeIAaYADzwBn6VIsRmGCXK7UZ8Kp2+6iUEbsh6VxnlpcFIbJVbMqoMn90Hu40y6OWoQ0='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1b0b6-bd30-77cc-a464-59af60053d52-0', tool_calls=[{'name': 'get_today_date', 'args': {}, 'id': 'call_2330809596147589008', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 675, 'output_tokens': 12, 'total_tokens': 687, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='2026-11-06', name='get_today_date', id='0039c792-3ae8-4cde-8322-464d56938e73

In [15]:
# Read every tool call from the returned message history.
tool_calls = []

for index, message in enumerate(messages):
    if not isinstance(message, AIMessage):
        continue

    for tool_call in message.tool_calls:
        print(f"{index + 1} th message says it needs {tool_call['name']} tool call")
        tool_output = None

        # Find the ToolMessage with the same tool-call ID.
        # After this AI decision to tool call lets check all messages to find the tool output (because tool is called actually after this message)
        for later_message in messages[index + 1 :]:
            if (
                isinstance(later_message, ToolMessage)
                and later_message.tool_call_id == tool_call["id"]
            ):
                tool_output = later_message.content
                break

        tool_calls.append(
            {
                "name": tool_call["name"],
                "arguments": tool_call["args"],
                "output": tool_output,
            }
        )

2 th message says it needs get_today_date tool call
6 th message says it needs get_today_date tool call
6 th message says it needs get_user_purchase_history tool call


In [14]:
tool_calls

[{'name': 'get_today_date', 'arguments': {}, 'output': '2026-11-06'},
 {'name': 'get_today_date', 'arguments': {}, 'output': '2026-11-06'},
 {'name': 'get_user_purchase_history',
  'arguments': {'user': 'user_10000000'},
  'output': '{"status": "there was no username found"}'}]

In [19]:
result["structured_response"].answer

'The username you provided was not valid. Could you please provide a valid username?'